# BlocoRocha A.01: Momentum Growth, annual rebalancing

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tiago-baptistalopes/bloco-rocha/blob/main/notebooks/A01_momentum_growth.ipynb)

| Item | Rule |
|---|---|
| Universe | Every United States and European firm with a market value of at least €1.5 billion at fund creation, 2005-01-25 (set `N_PER_REGION` to sample instead) |
| Entry | Top quartile 12-1 month momentum **and** top half price-to-book; filled to 50 names by combined rank |
| Exit | Debt-to-equity up 50% or more year on year, or price-to-book down 20% or more year on year |
| Rebalancing | Annual (entry, exit checks and equal reweighting) |
| Currency | Euro; United States dollar positions fully hedged (local returns, carry ignored) |
| Persona tests | Maximum drawdown no worse than −35%; value-at-risk-equivalent volatility 12–20% (Summary Risk Indicator class 4) |

Prices: yfinance. Fundamentals: Compustat via Wharton Research Data Services (cached to `data/fundamentals.csv`).

In [ ]:
%pip install -q yfinance quantstats wrds

In [ ]:
import os
import numpy as np
import pandas as pd
import yfinance as yf
import quantstats as qs
import matplotlib.pyplot as plt

START = pd.Timestamp('2005-01-25')
HISTORY_START = '2003-12-01'
N_HOLD = 50
N_PER_REGION = None
MIN_MCAP_EUR = 1.5e9
MOM_Q, PB_Q = 0.75, 0.50
DE_JUMP, PB_DROP = 0.50, -0.20
MDD_LIMIT, VEV_RANGE, RHP_YEARS = -0.35, (0.12, 0.20), 20
SEED = 42
WRDS_USERNAME = ''
DATA = 'data'
os.makedirs(DATA, exist_ok=True)

## 1. Universe: every firm above the size threshold at fund creation

All United States and European firms in Compustat with fiscal year 2004 data, pre-screened at a market value of €1 billion, then screened exactly at €1.5 billion on 2005-01-25 with yfinance prices. Compustat includes firms delisted since 2005; the share that yfinance cannot price measures the survivorship bias.

In [ ]:
def cached(name, fetch, dates=True):
    path = f'{DATA}/{name}.csv'
    if os.path.exists(path):
        return pd.read_csv(path, index_col=0, parse_dates=dates)
    df = fetch()
    df.to_csv(path)
    return df

CCYS = ['USD', 'GBP', 'CHF', 'SEK', 'DKK', 'NOK', 'PLN']

def fetch_fx():
    fx = yf.download([f'EUR{c}=X' for c in CCYS], start=HISTORY_START, auto_adjust=True, progress=False)['Close']
    fx.columns = [c[3:6] for c in fx.columns]
    return fx

fx = cached('fx', fetch_fx).ffill().bfill()
fx['EUR'] = 1.0

In [ ]:
EUROPE = ['GBR','FRA','DEU','NLD','BEL','LUX','IRL','ESP','PRT','ITA','CHE','AUT','SWE','DNK','NOR','FIN','GRC']
FUNDA_US = "f.indfmt='INDL' and f.datafmt='STD' and f.popsrc='D' and f.consol='C'"
FUNDA_EU = "f.indfmt='INDL' and f.datafmt='HIST_STD' and f.popsrc='I' and f.consol='C'"

def wrds_query(sql):
    import wrds
    db = wrds.Connection(wrds_username=WRDS_USERNAME or None)
    if not os.path.exists(os.path.expanduser('~/.pgpass')):
        db.create_pgpass_file()
    df = db.raw_sql(sql)
    db.close()
    return df

def fetch_compustat_universe():
    return wrds_query(f'''
        select f.gvkey, f.tic as symbol, 'US' as region, f.curcd, f.prcc_f * f.csho * 1e6 as mcap_local
        from comp.funda f join comp.company c on c.gvkey = f.gvkey
        where {FUNDA_US} and f.fyear = 2004 and c.loc = 'USA' and f.prcc_f * f.csho >= 1000
        union all
        select f.gvkey, s.isin as symbol, 'EU' as region, d.curcdd as curcd, d.prccd * d.cshoc as mcap_local
        from comp.g_funda f
        join comp.g_company c on c.gvkey = f.gvkey
        join comp.g_security s on s.gvkey = f.gvkey and s.iid = c.prirow
        join lateral (select prccd, cshoc, curcdd from comp.g_secd x
                      where x.gvkey = s.gvkey and x.iid = s.iid and x.datadate <= f.datadate and x.datadate > f.datadate - 10
                      order by x.datadate desc limit 1) d on true
        where {FUNDA_EU} and f.fyear = 2004 and c.loc in ({','.join(f"'{x}'" for x in EUROPE)})''').set_index('gvkey')

cu = cached('compustat_universe', fetch_compustat_universe, dates=False)
cu.index = cu.index.astype(str).str.zfill(6)
rate = fx.loc[:'2004-12-31'].iloc[-1]
cu['mcap_eur'] = cu['mcap_local'] / cu['curcd'].map(rate)
cu = cu[cu['mcap_eur'] >= 1e9]
cu['region'].value_counts()

In [ ]:
def yahoo_symbol(isin):
    try:
        quotes = yf.Search(isin, max_results=5).quotes
        return quotes[0]['symbol'] if quotes else None
    except Exception:
        return None

def fetch_yahoo_map():
    m = pd.Series({g: (s.replace('.', '-') if r == 'US' else yahoo_symbol(s)) for g, s, r in zip(cu.index, cu['symbol'], cu['region'])})
    return m.rename('ticker').to_frame()

ymap = cached('yahoo_map', fetch_yahoo_map, dates=False)['ticker'].dropna()
ymap.index = ymap.index.astype(str).str.zfill(6)
ymap = ymap[ymap.index.isin(cu.index) & ~ymap.duplicated()]
TICKERS = sorted(ymap)
REGION = pd.Series(cu.loc[ymap.index, 'region'].values, index=ymap.values)
GVKEY = pd.Series(ymap.index, index=ymap.values)

## 2. Prices and currencies (yfinance)

In [ ]:
def fetch_prices():
    return yf.download(TICKERS, start=HISTORY_START, auto_adjust=True, progress=False, threads=True)['Close']

def fetch_meta():
    rows = {}
    for t in TICKERS:
        try:
            rows[t] = yf.Ticker(t).fast_info['currency']
        except Exception:
            rows[t] = None
    return pd.Series(rows).rename('currency').to_frame()

px = cached('prices', fetch_prices).dropna(how='all', axis=1).dropna(how='all')
meta = cached('meta', fetch_meta, dates=False)
fx = fx.reindex(px.index).ffill().bfill()

def base_ccy(c):
    return 'GBP' if c in ('GBp', 'GBX') else c

def to_eur(t):
    c = meta.loc[t, 'currency']
    return px[t] * (0.01 if c in ('GBp', 'GBX') else 1.0) / fx[base_ccy(c)]

priced = [t for t in px.columns if t in meta.index and base_ccy(meta.loc[t, 'currency']) in fx.columns]
px_eur = pd.DataFrame({t: to_eur(t) for t in priced})
px_ret = pd.DataFrame({t: px[t] if meta.loc[t, 'currency'] == 'USD' else px_eur[t] for t in priced})

## 3. Fundamentals (Compustat via Wharton Research Data Services)

Debt = long-term debt + debt in current liabilities. Fiscal year $y-1$ is used at the January rebalance of year $y$ (accepted look-ahead bias). Compustat amounts are in millions.

In [ ]:
def fetch_fundamentals():
    keys = ','.join(f"'{g}'" for g in cu.index)
    return wrds_query(f'''
        select f.gvkey, f.fyear, f.curcd, f.ceq * 1e6 as ceq, (coalesce(f.dltt,0) + coalesce(f.dlc,0)) * 1e6 as debt,
               f.csho * 1e6 as shares
        from comp.funda f where {FUNDA_US} and f.fyear between 2002 and 2026 and f.gvkey in ({keys})
        union all
        select f.gvkey, f.fyear, f.curcd, f.ceq * 1e6, (coalesce(f.dltt,0) + coalesce(f.dlc,0)) * 1e6, d.cshoc
        from comp.g_funda f
        join comp.g_company c on c.gvkey = f.gvkey
        left join lateral (select cshoc from comp.g_secd x
                           where x.gvkey = f.gvkey and x.iid = c.prirow and x.datadate <= f.datadate and x.datadate > f.datadate - 10
                           order by x.datadate desc limit 1) d on true
        where {FUNDA_EU} and f.fyear between 2002 and 2026 and f.gvkey in ({keys})''').set_index('gvkey')

fund = cached('fundamentals', fetch_fundamentals, dates=False).reset_index()
fund['gvkey'] = fund['gvkey'].astype(str).str.zfill(6)
tick = pd.Series(GVKEY.index, index=GVKEY.values)
fund['ticker'] = fund['gvkey'].map(tick)
fund = fund.dropna(subset=['ticker', 'ceq']).drop_duplicates(['ticker', 'fyear'], keep='last')
ceq, debt, shares = (fund.pivot(index='fyear', columns='ticker', values=v) for v in ('ceq', 'debt', 'shares'))
curcd = fund.groupby('ticker')['curcd'].last()
de = (debt / ceq).where(ceq > 0)

## 4. Exact size screen at fund creation and annual signals

In [ ]:
pxf = px_eur.ffill(limit=10)
pxr = px_ret.ffill(limit=10)
DATES = [px.index[px.index >= pd.Timestamp(y, 1, 25)][0] for y in range(START.year, px.index[-1].year + 1)
         if (px.index >= pd.Timestamp(y, 1, 25)).any()]

def at(df, d):
    return df.loc[:d].iloc[-1]

def book_eur(d):
    y = d.year - 1
    return ceq.reindex([y]).iloc[0] / curcd.reindex(ceq.columns).map(at(fx, d)).astype(float)

def mcap_eur(d):
    return at(pxf, d).reindex(shares.columns) * shares.reindex([d.year - 1]).iloc[0]

def pb(d):
    b = book_eur(d)
    return (mcap_eur(d) / b).where(b > 0)

def momentum(d):
    return at(pxr, d - pd.DateOffset(months=1)) / at(pxr, d - pd.DateOffset(months=12)) - 1

mcap0 = mcap_eur(DATES[0]).dropna()
UNIVERSE = sorted(mcap0[mcap0 >= MIN_MCAP_EUR].index)
if N_PER_REGION:
    rng = np.random.default_rng(SEED)
    UNIVERSE = [t for r in ('US', 'EU') for t in rng.choice(sorted(t for t in UNIVERSE if REGION[t] == r),
                size=min(N_PER_REGION, sum(REGION[t] == r for t in UNIVERSE)), replace=False)]

funnel = pd.DataFrame({
    'Compustat, pre-screen above €1bn': cu['region'].value_counts(),
    'Found on Yahoo Finance': REGION.value_counts(),
    'Priced on 2005-01-25': REGION[at(pxf, DATES[0]).dropna().index.intersection(REGION.index)].value_counts(),
    'Universe (above €1.5bn)': REGION[UNIVERSE].value_counts()}).T
funnel['Total'] = funnel.sum(axis=1)
funnel

## 5. Portfolio rules

At each annual date $t$: holdings that trigger an exit signal are sold,
$$\frac{D/E_{y-1}}{D/E_{y-2}} - 1 \ge 0.5 \quad \text{or} \quad \frac{P/B_t}{P/B_{t-1}} - 1 \le -0.2,$$
the remaining holdings are kept, and free slots are filled first with names in the top momentum quartile and the top price-to-book half, then with the best remaining combined rank $\tfrac{1}{2}\left(\text{rank}_{\text{momentum}} + \text{rank}_{P/B}\right)$. All holdings are reset to equal weight.

In [ ]:
holdings, log = {}, []
held, prev_pb = [], None
for d in DATES:
    alive = at(pxf, d).reindex(UNIVERSE).dropna().index
    mom, pbs = momentum(d).reindex(alive), pb(d).reindex(alive)
    y = d.year - 1
    de_jump = (de.loc[y] / de.loc[y - 1] - 1 >= DE_JUMP) if {y, y - 1} <= set(de.index) else pd.Series(False, index=de.columns)
    pb_drop = (pbs / prev_pb - 1 <= PB_DROP) if prev_pb is not None else pd.Series(False, index=alive)
    exits = [t for t in held if t not in alive or de_jump.get(t, False) or pb_drop.get(t, False)]
    held = [t for t in held if t not in exits]
    ok = mom.dropna().index.intersection(pbs.dropna().index)
    score = (mom[ok].rank(pct=True) + pbs[ok].rank(pct=True)) / 2
    passed = (mom[ok] >= mom[ok].quantile(MOM_Q)) & (pbs[ok] >= pbs[ok].quantile(PB_Q))
    order = pd.DataFrame({'passed': passed, 'score': score}).sort_values(['passed', 'score'], ascending=False).index
    entries = [t for t in order if t not in held][:max(N_HOLD - len(held), 0)]
    held = held + entries
    holdings[d] = held
    prev_pb = pbs
    log.append({'date': d.date(), 'holdings': len(held), 'entries': len(entries), 'exits': len(exits),
                'passed_both_filters': int(passed.sum()), 'exit_leverage': sum(bool(de_jump.get(t, False)) for t in exits),
                'exit_price_to_book': sum(bool(pb_drop.get(t, False)) for t in exits)})
pd.DataFrame(log).set_index('date')

## 6. Backtest

In [ ]:
def backtest(holdings, prices):
    dates = list(holdings) + [prices.index[-1]]
    parts = []
    for d0, d1 in zip(dates[:-1], dates[1:]):
        p = prices.loc[d0:d1, holdings[d0]].ffill()
        parts.append((p / p.iloc[0]).mean(axis=1).pct_change().dropna())
    return pd.concat(parts)

fund_r = backtest(holdings, px_ret).rename('BlocoRocha A.01')
bench_holdings = {d: list(at(pxf, d).reindex(UNIVERSE).dropna().index) for d in DATES}
bench_r = backtest(bench_holdings, px_ret).rename('Universe equal weight')

cum = (1 + pd.concat([fund_r, bench_r], axis=1)).cumprod()
cum = pd.concat([pd.DataFrame(1.0, index=[DATES[0]], columns=cum.columns), cum])
ax = cum.plot(figsize=(11, 5), logy=True, title='Cumulative returns (start = 1, log scale)')
ax.axhline(1, color='grey', lw=0.8)
plt.show()
cum.iloc[-1]

In [ ]:
qs.reports.metrics(fund_r, benchmark=bench_r, mode='full', display=True)

## 7. Persona tests

Value-at-risk-equivalent volatility (Commission Delegated Regulation (EU) 2017/653, Annex II), with $\sigma$, $\mu_1$ (skewness) and $\mu_2$ (excess kurtosis) of daily log returns, $N = 256\,T$ trading days over the recommended holding period $T$:
$$\text{VaR}_{\text{return}} = \sigma\sqrt{N}\left(-1.96 + 0.474\frac{\mu_1}{\sqrt{N}} - 0.0687\frac{\mu_2}{N} + 0.146\frac{\mu_1^2}{N}\right) - 0.5\,\sigma^2 N, \qquad \text{VEV} = \frac{\sqrt{3.842 - 2\,\text{VaR}_{\text{return}}} - 1.96}{\sqrt{T}}$$

In [ ]:
def vev(r, T=RHP_YEARS):
    x = np.log1p(r.dropna())
    s, m1, m2, N = x.std(), x.skew(), x.kurt(), 256 * T
    var = s * np.sqrt(N) * (-1.96 + 0.474 * m1 / np.sqrt(N) - 0.0687 * m2 / N + 0.146 * m1**2 / N) - 0.5 * s**2 * N
    return (np.sqrt(3.842 - 2 * var) - 1.96) / np.sqrt(T)

def persona_tests(r):
    mdd, v = qs.stats.max_drawdown(r), vev(r)
    return pd.DataFrame({
        'value': [mdd, v, qs.stats.volatility(r), qs.stats.value_at_risk(r)],
        'target': [f'>= {MDD_LIMIT:.0%}', f'{VEV_RANGE[0]:.0%} to {VEV_RANGE[1]:.0%}', 'reference', 'reference (daily, 95%)'],
        'pass': [mdd >= MDD_LIMIT, VEV_RANGE[0] <= v <= VEV_RANGE[1], None, None]},
        index=['Maximum drawdown', 'Value-at-risk-equivalent volatility', 'Annualised volatility', 'Historical value at risk'])

tests = persona_tests(fund_r)
print('Persona conditions met:', bool(tests['pass'].dropna().all()))
tests